In [0]:

# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_departments
# Source          : departments.csv
# Target          : procurement.bronze.bronze_departments
# Audit Table     : procurement.audit.duplicate_departments
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw department master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load Department master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Department IDs and store them in the Audit schema
# for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source
# for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_DEPARTMENTS)
print(AUDIT_DUPLICATE_DEPARTMENTS)
print(DEPARTMENTS_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)

In [0]:
# ============================================================
# Read Departments CSV (Test)
# ============================================================

test_df = (
    spark.read
         .format("csv")
         .option("header", True)
         .load(DEPARTMENTS_FILE)
)

display(test_df)
print(test_df.columns)

In [0]:
#Departments schema
departments_schema = StructType([
    StructField("department_id", StringType(), False),
    StructField("department_name", StringType(), True),
    StructField("division", StringType(), True),
    StructField("region", StringType(), True),
    StructField("cost_center", StringType(), True),
    StructField("annual_budget_usd", DecimalType(18,2), True)
])
# Read Departments master data from landing volume
bronze_departments_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(departments_schema)
    .load(DEPARTMENTS_FILE)
)
# Display preview Departments data
display(bronze_departments_df)

#Source Data validation 
print(f"Total Records : {bronze_departments_df.count()}")

print("\nSchema:")
bronze_departments_df.printSchema()

print("\nColumns:")
print(bronze_departments_df.columns)